In [1]:
from pathlib import Path
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

def concat_large_parquets(dir_path: str | Path) -> pd.DataFrame:
    path = Path(dir_path).resolve()
    parquet_files = sorted(list(path.glob("*.parquet")))
    total_files = len(parquet_files)
    if total_files == 0:
        raise FileNotFoundError(f"No .parquet files found in directory: {path}")
    print(f"Starting concatenation for {total_files} file(s) in '{path.name}'...\n")

    target_schema = pa.schema([
        ('year', pa.int64()),
        ('language_type', pa.string()),
        ('text', pa.string()),
        ('decade', pa.int64()),
        ('category', pa.string()),
        ('text_length', pa.int64())
    ])
    arrow_tables = []
    for idx, file in enumerate(parquet_files, start=1):
        table = pq.read_table(file)
        table = table.select(target_schema.names)
        table = table.cast(target_schema)
        arrow_tables.append(table)
        print(f"[{idx}/{total_files}] Successfully loaded and unified: {file.name} ({table.num_rows:,} rows)")
    print("\nConcatenating all tables in memory...")
    combined_table = pa.concat_tables(arrow_tables)
    df = combined_table.to_pandas()

    print(f"Done! Combined {total_files} files with a total of {len(df):,} rows.")
    return df

df = concat_large_parquets("../.idea/data/processed_dbs")
df.to_parquet("../processed_dbs/combined_dataset.parquet")

Starting concatenation for 7 file(s) in 'processed_dbs'...

[1/7] Successfully loaded and unified: common_corpus.parquet (1,491,184 rows)
[2/7] Successfully loaded and unified: LoC-PD-Books.parquet (26,585,193 rows)
[3/7] Successfully loaded and unified: pre_1900_corpus_40.parquet (8,228,496 rows)
[4/7] Successfully loaded and unified: reddit.parquet (15,000 rows)
[5/7] Successfully loaded and unified: textage_large.parquet (91,728 rows)
[6/7] Successfully loaded and unified: twitter.parquet (5,086 rows)
[7/7] Successfully loaded and unified: youtube.parquet (15,100 rows)

Concatenating all tables in memory...
Done! Combined 7 files with a total of 36,431,787 rows.


In [2]:
df.shape

(36431787, 6)